# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring**

I picked this lane because the starter pipeline already demonstrates a 3x improvement over a hand-written rule: random forest achieves Precision@50 of 0.740 versus the baseline's 0.240 (37 vs 12 correct pages out of the top 50). That gap is large enough to investigate whether a learned ranking genuinely finds more declining pages — and whether reason codes and confidence labels make those recommendations useful to a human reviewer. The starter also defines a clear future-looking improvement path: replace the current proxy label (`trend_direction == "down"`, measured from the current window) with a true forward-looking one (features from prior 90 days → decline over the next 30 days), which the warehouse data supports.

In [9]:
# (Markdown answer above — this cell reserved for code cells that support section 1)
# No supporting code needed here; section 3 has the data look.


## 2. The question: decision, action, cost of a wrong call

- **Decision improved:** Which existing content page should an editor review first for refresh?
- **Who acts:** A content editor or SEO manager with limited weekly capacity (e.g. 20–50 pages per cycle).
- **Action they take:** Open the page, inspect it against the reason codes, decide whether to refresh, expand, rewrite metadata, or monitor.
- **Cost of a wrong call:**
  - *False positive* (recommend a page that does not need work): wasted editor hours — reviewing a healthy page instead of a declining one. At ~15 minutes per review, 10 false positives cost 2.5 hours.
  - *False negative* (miss a truly declining page): continued traffic loss that could have been caught. The page keeps decaying, and the missed opportunity compounds over time.
- **Why ML helps:** The signal is spread across many columns — impressions, position, age, freshness, CTR, engagement rates, content type — and the patterns are too messy to capture with a handful of fixed rules. The hand-written baseline got only 12/50 right; the random forest got 37/50, suggesting learned combinations of these signals outperform simple thresholds.

In [10]:
# (Markdown answer above — this cell reserved for code cells that support section 2)
# No supporting code needed here.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

**Number 1 — Declining pages are common:** 54.2% of the 30,000 rows have `trend_direction == "down"` (16,262 pages). This is not a rare-event problem; there is enough signal to learn from.

**Number 2 — Baseline vs. model gap:** The hand-written rule baseline achieves Precision@50 of 0.240 (12/50 correct). The random forest reaches 0.740 (37/50 correct) — a 3x improvement on the top-50 ranking quality, validated on held-out clients.

**Number 3 — Enough visibility to matter:** Median impressions per page is 731 across 32 clients. Most pages have real search exposure worth protecting. 1,205 rows have no position data (`avg_position == 0`), which is a known gotcha to handle, not a blocker.

In [11]:
import pandas as pd
import json

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f'Total rows: {len(df):,}')
print(f'Clients: {df.client_id.nunique()}')
print(f'Declining rate: {(df.trend_direction.str.lower() == "down").mean():.3f}')
print(f'Median impressions: {df.impressions_90d.median():.0f}')
print(f'Median sessions (pages with >0): {df[df.sessions_90d > 0].sessions_90d.median():.0f}')
print()

results = json.load(open('../../outputs/model_results.json'))
print('Model Precision@50:')
for m, v in results['models'].items():
    print(f'  {m}: {v["precision_at_50"]:.3f}')
print(f'  baseline: {results["baseline"]["baseline_precision_at_50"]:.3f}')
print(f'Best model: {results["best_model"]["name"]}')
print(f'Validation: {results["split_strategy"]}')
print(f'Target positive rate: {results["target_positive_rate"]:.3f}')


Total rows: 30,000
Clients: 32
Declining rate: 0.542
Median impressions: 731
Median sessions (pages with >0): 7

Model Precision@50:
  decision_tree: 0.620
  logistic_regression: 0.400
  random_forest: 0.740
  baseline: 0.240
Best model: random_forest
Validation: client_holdout
Target positive rate: 0.542


## 4. Careful words: what I can and can't claim

**I can claim (observed, directional, decision-support):**
- The starter dataset shows a measurable association between certain observed signals (days with impressions, log impressions, avg position, content age) and the `trend_direction == "down"` label.
- A random forest ranked queue achieves higher Precision@50 on held-out clients than a hand-written baseline rule — on this 30,000-row slice.
- Reason codes and confidence labels can help an editor triage which pages to review first.

**I cannot claim:**
- That refreshing a recommended page *causes* a recovery (no experiment, no causal design).
- That the model "predicts Google's algorithm" (it uses observed search and analytics signals, not algorithm internals).
- That the starter label (`trend_direction == "down"`) is a true future-outcome target (it is a current-window proxy).
- That results on the 30k-row starter slice generalize to the full 79M-row warehouse without re-validation.

In [12]:
# No extra code needed for this section — the claims above are backed by the numbers in section 3.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.